# Create Magnus Ehrnrooth Foundation Awards (Magnus Ehrnroothin säätiö)

Creates the grants of the Magnus Ehrnrooth Foundation (Magnus Ehrnroothin säätiö /
Magnus Ehrnrooths stiftelse) from the foundation's own awarded-grants listings.

**Prerequisite:** `scripts/local/magnus_ehrnrooth_to_s3.py` (uploads the parquet).

**Source:** https://magnusehrnroothinsaatio.fi/en/grants/awarded-grants/ is built from
WordPress posts of type `myonnetyt_apurahat`, one per (grant year x field), read from
the site's public REST API (`/wp-json/wp/v2/myonnetyt_apurahat`, English versions; the
Finnish and Swedish versions list the same grants). Each grant gives degree + name,
amount (EUR) and project title. Every field-year sum reconciles with the total printed
on the listing page. research.fi (ladder item 0) does not carry this foundation.

**277 grants, 2024-2026, EUR 7.16M; 100% title and amount, 99.6% named grantee**
(local run 2026-10-01). Older lists (2010-2023) were blog posts and PDFs on the
previous site; the live site no longer serves them (the PDF links in the surviving
2019-2023 announcement posts return 404), so they are not included here.

**Scope:** all six fields are kept: Physics, Chemistry, Medicinal Chemistry,
Mathematics, Astronomy, and Education (13 school-teaching / science-education grants:
teacher training days, school science competitions, science camps; kept and flagged
under the batch scope rule, `funding_type = 'grant'`, scheme 'Education'). One grant
goes to an association (Suomen Tähtitieteilijäseura ry): no lead investigator.

**Field mapping:** `display_name` = project title; `funder_scheme` = field of science;
`start_year` = grant year (decisions each March); dates NULL (not published);
`lead_investigator` = grantee, name isolated from the printed degree (the trailing
run of name tokens) and split by the canonical `split_name`; 6 family-first listings
are put given-first. No affiliation is printed. Amounts EUR.

**`funder_award_id` (2.1.1):** none is published. Citing works write internal project
numbers (`ma2014n1`, `KE2012n8`, `ME2012n44` style) in some of the 99 acknowledgement
shell rows; the listings never show them, so a synthetic
`MEF-<year>-<field>-<family>-<given>` key is used; 0 shells collapse.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320322400`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 02cdzet69, doi 10.13039/501100004155)

**Priority:** `569` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.magnus_ehrnrooth_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/magnus_ehrnrooth/magnus_ehrnrooth_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_ids,
       MIN(grant_year) as min_year, MAX(grant_year) as max_year, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.magnus_ehrnrooth_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.magnus_ehrnrooth_raw;

In [ ]:
%sql
SELECT grant_year, field, COUNT(*) AS n, ROUND(SUM(amount), 0) AS eur, collect_set(field_total_check) AS total_check
FROM openalex.awards.magnus_ehrnrooth_raw GROUP BY 1, 2 ORDER BY 1, 2;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320322400 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320322400;

## Step 2: Create Magnus Ehrnrooth Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.magnus_ehrnrooth_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320322400  -- Magnus Ehrnroothin Säätiö
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    COALESCE(NULLIF(TRIM(g.title), ''),
             CONCAT('Magnus Ehrnrooth Foundation grant: ', COALESCE(g.grantee, g.grantee_raw), ' (', g.grant_year, ')')) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    'EUR' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.field = 'Education' THEN 'grant' ELSE 'research' END as funding_type,
    NULLIF(TRIM(g.field), '') as funder_scheme,
    'magnus_ehrnrooth_awarded_grants' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.grant_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>)
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.magnus_ehrnrooth_raw g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

## Shape check (2.1.1)
Synthetic `MEF-<year>-<field slug>-<family slug>-<given slug>[-b..]` keys only.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 0, 'unexpected funder_award_id shape') AS shape_ok
FROM openalex.awards.magnus_ehrnrooth_awards
WHERE NOT funder_award_id RLIKE '^MEF-(19|20)[0-9]{2}-[a-z0-9-]+$';

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.magnus_ehrnrooth_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'magnus_ehrnrooth_awarded_grants' AND priority = 569;

-- Priority 569: direct-from-funder ingest of the foundation's own awarded-grants listings.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    569 as priority
FROM openalex.awards.magnus_ehrnrooth_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, start_year,
       lead_investigator.given_name, lead_investigator.family_name
FROM openalex.awards.magnus_ehrnrooth_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.magnus_ehrnrooth_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.magnus_ehrnrooth_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.magnus_ehrnrooth_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.magnus_ehrnrooth_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator) as has_pi
FROM openalex.awards.magnus_ehrnrooth_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'magnus_ehrnrooth_awarded_grants'
GROUP BY provenance, priority;